In [1]:
! pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 719.8/719.8 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.1/124.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.9/246.9 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 46.0 MB/s eta 0:00:00


In [2]:
from fastai.vision.all import *
from fastbook import *

matplotlib.rc('image', cmap='Greys')

In [3]:
path = untar_data(URLs.MNIST_SAMPLE)

<div><progress max="3214948" value="3219456"></progress> 100.14% [3219456/3214948 00:01&lt;00:00]</div>

In [4]:
Path.BASE_PATH = path

In [5]:
path.ls()

[Path('valid'), Path('labels.csv'), Path('train')]

In [6]:
(path / 'train').ls()

[Path('train/7'), Path('train/3')]

In [7]:
threes = (path/'train/3').ls().sorted()
sevens = (path/'train/7').ls().sorted()

threes

(#6131) [Path('train/3/10.png'), Path('train/3/10000.png'), Path('train/3/10011.png'), Path('train/3/10031.png'), Path('train/3/10034.png'), Path('train/3/10042.png'), Path('train/3/10052.png'), Path('train/3/1007.png'), Path('train/3/10074.png'), Path('train/3/10091.png'), Path('train/3/10093.png'), Path('train/3/10097.png'), Path('train/3/10099.png'), Path('train/3/10116.png'), Path('train/3/10125.png'), Path('train/3/10137.png'), Path('train/3/10141.png'), Path('train/3/10144.png'), Path('train/3/10155.png'), Path('train/3/10161.png'), Path('train/3/10206.png'), Path('train/3/1021.png'), Path('train/3/10210.png'), Path('train/3/10214.png'), Path('train/3/10238.png'), Path('train/3/10260.png'), Path('train/3/10278.png'), Path('train/3/10282.png'), Path('train/3/10314.png'), Path('train/3/10322.png'), Path('train/3/10328.png'), Path('train/3/10329.png'), Path('train/3/10330.png'), Path('train/3/10349.png'), Path('train/3/1035.png'), Path('train/3/10360.png'), Path('train/3/10369.png')

In [8]:
Image.open(sevens[0]).to_thumb(128, 128)

In [9]:
threes_tensors = [tensor(Image.open(p)) for p in threes]
sevens_tensors = [tensor(Image.open(p)) for p in sevens]

len(threes_tensors), len(sevens_tensors)

(6131, 6265)

In [10]:
threes_tensors[0].shape

torch.Size([28, 28])

In [11]:
stacked_threes = torch.stack(threes_tensors).float()/255
stacked_sevens = torch.stack(sevens_tensors).float()/255

stacked_threes.shape, stacked_sevens.shape

(torch.Size([6131, 28, 28]), torch.Size([6265, 28, 28]))

In [12]:
valid_threes = [tensor(Image.open(p)) for p in (path/'valid/3').ls().sorted()]
valid_sevens = [tensor(Image.open(p)) for p in (path/'valid/7').ls().sorted()]

valid_3_tens = torch.stack(valid_threes).float()/255
valid_7_tens = torch.stack(valid_sevens).float()/255

valid_3_tens.shape, valid_7_tens.shape

(torch.Size([1010, 28, 28]), torch.Size([1028, 28, 28]))

### The MNIST Loss Function

In [13]:
# represent the images as a list of vectors, where each vector is the image matrix
# .view() changes the shape without changing the contents
train_x = torch.cat([stacked_threes, stacked_sevens]).view(-1, 28*28)
train_x

tensor([[0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        ...,
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.],
        [0., 0., 0.,  ..., 0., 0., 0.]])

In [14]:
# now get the labels, 1 for threes and 0 for sevens
train_y = tensor([1]*len(threes) + [0]*len(sevens)).unsqueeze(1)
train_x.shape, train_y.shape

(torch.Size([12396, 784]), torch.Size([12396, 1]))

In [15]:
# convert to dataset-compatible shape that returns (x,y) values for pytorch
dset = list(zip(train_x, train_y))
x,y = dset[0]
x.shape,y

(torch.Size([784]), tensor([1]))

In [37]:
valid_x = torch.cat([valid_3_tens, valid_7_tens]).view(-1, 28*28)
valid_y = tensor([1]*len(valid_threes) + [0]*len(valid_sevens)).unsqueeze(1)

valid_x.shape, valid_y.shape

(torch.Size([2038, 784]), torch.Size([2038, 1]))

In [39]:
valid_dset = list(zip(valid_x, valid_y))

valid_dset[0][0].shape, valid_dset[0][1]

(torch.Size([784]), tensor([1]))

Now set an initial random weight for every pixel. This is step of the SGD process.

In [42]:
def init_params(size, std=1.0):
    return (torch.randn(size) * std).requires_grad_()

In [67]:
weights = init_params((28*28,1))
weights

tensor([[ 1.7746e+00],
        [ 1.5162e+00],
        [-1.0466e+00],
        [ 1.3759e+00],
        [ 5.7629e-02],
        [ 3.5209e-01],
        [ 1.0967e+00],
        [-9.8828e-01],
        [-5.6178e-01],
        [-4.7884e-01],
        [-7.5233e-01],
        [-4.7687e-01],
        [ 1.2205e+00],
        [-1.5345e-01],
        [-5.1195e-01],
        [ 1.1924e-01],
        [ 2.3616e-02],
        [-7.1655e-01],
        [ 9.3131e-01],
        [ 1.1275e-01],
        [-2.9633e-01],
        [ 6.2816e-01],
        [ 7.9721e-01],
        [ 6.9573e-01],
        [ 4.5539e-01],
        [ 5.5991e-01],
        [ 2.1209e-01],
        [ 1.1861e+00],
        [-8.8241e-02],
        [-1.1839e+00],
        [-2.3502e-01],
        [ 6.6807e-01],
        [ 1.3330e+00],
        [ 1.2649e-01],
        [-1.5771e-01],
        [ 5.7832e-01],
        [ 5.1833e-01],
        [ 1.0639e-01],
        [-7.0810e-02],
        [ 1.8684e-01],
        [-8.3579e-01],
        [-6.9006e-01],
        [ 4.6465e-01],
        [ 9

In [45]:
bias = init_params(1)
bias

tensor([0.3215], requires_grad=True)

In [74]:
(dset[0][0]*weights).sum() + bias

tensor([1087.5582], grad_fn=<AddBackward0>)

In [87]:
train_x[0].shape

torch.Size([784])

In [86]:
weights.T.shape, weights.shape

(torch.Size([1, 784]), torch.Size([784, 1]))

In [71]:
(train_x[0]*weights.T).sum() + bias

tensor([13.7230], grad_fn=<AddBackward0>)

In [89]:
def linear1(xb):
    return xb@weights + bias

preds = linear1(train_x)
preds, preds.shape

(tensor([[13.7230],
         [21.9207],
         [21.3904],
         ...,
         [ 5.1306],
         [ 9.9300],
         [10.0229]], grad_fn=<AddBackward0>),
 torch.Size([12396, 1]))

In [99]:
corrects = (preds>0.0).float() == train_y
corrects

tensor([[ True],
        [ True],
        [ True],
        ...,
        [False],
        [False],
        [False]])

In [101]:
corrects.float().mean().item()

0.49249759316444397

In [102]:
with torch.no_grad(): weights[0] *= 1.0001

In [103]:
preds = linear1(train_x)
((preds>0.0).float() == train_y).float().mean().item()

0.49249759316444397